In [ ]:
!nvidia-smi
# fresh clone:
!git clone https://github.com/iesxz-c/Final.git
%cd Final
# ...or if Final/ already exists:
# %cd Final
# !git pull origin master
!git log --oneline -3

Mon Sep 14 15:49:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [ ]:
!pip install -q transformers huggingface_hub safetensors pyyaml

In [ ]:
!find /content/drive/MyDrive -type f \( -name "activities.json" -o -name "ucf_events.json" -o -name "detections.json" \) 2>/dev/null

In [ ]:

# Cell 4 - Point the project at Drive. Outputs also live on Drive so a
# Colab interruption never destroys checkpoints.
import os
import pathlib
DATASET_ROOT = '/content/drive/MyDrive'  # <-- EDIT if your folders live elsewhere
ANOMALY_ROOT = f'{DATASET_ROOT}/Anomaly-Videos-Part-1/Anomaly-Videos-Part-1'
NORMAL_ROOT = f'{DATASET_ROOT}/Normal_Videos_for_Event_Recognition/Normal_Videos_for_Event_Recognition'
SPLIT_ROOT = '/content/Final/UCF_Crimes-Train-Test-Split/Action_Regnition_splits'
OUTPUT_ROOT = f'{DATASET_ROOT}/ucf-crime-output/phase2c'
os.environ['CCTV_ANOMALY_VIDEOS_DIR'] = ANOMALY_ROOT
os.environ['CCTV_NORMAL_VIDEOS_DIR'] = NORMAL_ROOT
assert pathlib.Path(ANOMALY_ROOT).is_dir(), f'missing: {ANOMALY_ROOT}'
assert pathlib.Path(NORMAL_ROOT).is_dir(), f'missing: {NORMAL_ROOT}'
assert pathlib.Path(SPLIT_ROOT).is_dir(), f'missing: {SPLIT_ROOT} (pull latest repo)'
print('anomaly:', ANOMALY_ROOT)
print('normal :', NORMAL_ROOT)
print('splits :', SPLIT_ROOT)
print('output :', OUTPUT_ROOT)

anomaly: /content/drive/MyDrive/Anomaly-Videos-Part-1/Anomaly-Videos-Part-1
normal : /content/drive/MyDrive/Normal_Videos_for_Event_Recognition/Normal_Videos_for_Event_Recognition
splits : /content/Final/UCF_Crimes-Train-Test-Split/Action_Regnition_splits
output : /content/drive/MyDrive/ucf-crime-output/phase2c


In [ ]:
!python scripts/health_check.py
!python -m src.pipeline.inventory
!python -m src.pipeline.select_subset

Health check - /content/Final/Final/Final/Final

== Python ==
  version: 3.13.15
  [PASS]

== Dependencies ==
  pyyaml: 6.0.3 [PASS]

== Project layout ==
  src: [PASS]
  src/pipeline: [PASS]
  src/evidence: [PASS]
  src/agents: [PASS]
  src/agents/query_planning: [PASS]
  src/agents/evidence_retrieval: [PASS]
  src/agents/temporal_correlation: [PASS]
  src/agents/verification_report: [PASS]
  config: [PASS]
  scripts: [PASS]
  data: [FAIL] missing

== Configuration ==
  loaded: /content/Final/Final/Final/Final/config/config.example.yaml
  project root: /content/Final/Final/Final/Final
  [PASS]

== External datasets (informational) ==
  anomaly_videos: /content/drive/MyDrive/Anomaly-Videos-Part-1/Anomaly-Videos-Part-1 [PASS]
  normal_videos: /content/drive/MyDrive/Normal_Videos_for_Event_Recognition/Normal_Videos_for_Event_Recognition [PASS]

Summary: FAILURES detected
Scanning anomaly: /content/drive/MyDrive/Anomaly-Videos-Part-1/Anomaly-Videos-Part-1
  [anomaly] probed 100/849
  [ano

In [ ]:
!python -m src.pipeline.extract_ucf_events --limit-videos 2 --max-windows 5 --device cuda

preprocessor_config.json: 100% 415/415 [00:00<00:00, 2.39MB/s]
config.json: 100% 1.45k/1.45k [00:00<00:00, 983kB/s]

model.safetensors: downloading bytes:  16% 192M/1.22G [00:01<00:04, 245MB/s, 13.6MB/s  ]
model.safetensors: reconstructing file:   6% 67.0M/1.22G [00:01<00:31, 36.4MB/s]
model.safetensors: downloading bytes:  27% 326M/1.22G [00:02<00:03, 261MB/s, 27.9MB/s  ]
model.safetensors: downloading bytes:  95% 1.16G/1.22G [00:04<00:00, 283MB/s, 92.8MB/s  ]
model.safetensors: reconstructing file:  44% 536M/1.22G [00:05<00:07, 93.1MB/s, 24.8MB/s  ]
model.safetensors: downloading bytes: 100% 1.16G/1.16G [00:07<00:00, 149MB/s, 92.4MB/s  ]
model.safetensors: reconstructing file: 100% 1.22G/1.22G [00:07<00:00, 156MB/s, 92.5MB/s  ]
Event model: OPear/videomae-large-finetuned-UCF-Crime (bc5f1c1058158a05f8710de2b7c7c372fe69062b) on cuda (16 frames @ 8.0fps, top-5)
[1/2] anomaly/Assault/Assault001_x264.mp4
  5 windows (video 30.00fps)
[2/2] anomaly/Assault/Assault002_x264.mp4
  5 windows (v

In [ ]:
!python -m src.pipeline.extract_ucf_events --device cuda

Event model: OPear/videomae-large-finetuned-UCF-Crime (bc5f1c1058158a05f8710de2b7c7c372fe69062b) on cuda (16 frames @ 8.0fps, top-5)
[1/40] anomaly/Assault/Assault001_x264.mp4
  39 windows (video 30.00fps)
[2/40] anomaly/Assault/Assault002_x264.mp4
  40 windows (video 30.00fps)
[3/40] anomaly/Assault/Assault003_x264.mp4
  70 windows (video 30.00fps)
[4/40] anomaly/Assault/Assault004_x264.mp4
  51 windows (video 30.00fps)
[5/40] anomaly/Assault/Assault005_x264.mp4
  20 windows (video 30.00fps)
[6/40] anomaly/Fighting/Fighting002_x264.mp4
  42 windows (video 30.00fps)
[7/40] anomaly/Fighting/Fighting003_x264.mp4
  49 windows (video 30.00fps)
[8/40] anomaly/Fighting/Fighting004_x264.mp4
  263 windows (video 30.00fps)
[9/40] anomaly/Fighting/Fighting005_x264.mp4
  28 windows (video 30.00fps)
[10/40] anomaly/Fighting/Fighting006_x264.mp4
  15 windows (video 30.00fps)
[11/40] anomaly/Robbery/Robbery001_x264.mp4
  16 windows (video 30.00fps)
[12/40] anomaly/Robbery/Robbery002_x264.mp4
  51 wi

In [ ]:
%cd /content/Final
!git pull origin master
!git log --oneline -1

/content/Final
remote: Enumerating objects: 14, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 9 (delta 3), reused 9 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (9/9), 10.35 KiB | 1.48 MiB/s, done.
From https://github.com/iesxz-c/Final
 * branch            master     -> FETCH_HEAD
   6426900..87183ec  master     -> origin/master
Updating 6426900..87183ec
Fast-forward
 src/evidence/fusion.py        | 434 ++++++++++++++++++++++++++++++++++++++++++
 src/pipeline/fuse_evidence.py | 167 ++++++++++++++++
 tests/test_fusion.py          | 231 ++++++++++++++++++++++
 3 files changed, 832 insertions(+)
 create mode 100644 src/evidence/fusion.py
 create mode 100644 src/pipeline/fuse_evidence.py
 create mode 100644 tests/test_fusion.py
87183ec (HEAD -> master, origin/master, origin/HEAD) Phase 2D: evidence fusion layer (alignment, incidents, scores) + tests


In [ ]:
!ls data/evidence/ data/experiments/ 2>/dev/null
!ls /content/drive/MyDrive/ucf-crime-output/phase2b/ 2>/dev/null
!ls -d /content/drive/MyDrive/ucf-crime-output/phase2c_ucf* 2>/dev/null

data/evidence/:
phase2c_ucf

data/experiments/:
phase2_subset.json


In [ ]:
!ls /content/drive 2>&1 | head -2
!ls data/evidence/phase2c_ucf/
!python -c "import json; m=json.load(open('data/evidence/phase2c_ucf/manifest.json')); print('processed:', m.get('videos_processed'), '| observations:', m.get('observations'))"

MyDrive
manifest.json  ucf_events.json	videos.json
processed: 40 | observations: 1931


In [ ]:
!python -m src.pipeline.extract_activity --device cuda


preprocessor_config.json: 100% 271/271 [00:00<00:00, 1.50MB/s]
config.json: 100% 22.9k/22.9k [00:00<00:00, 58.5MB/s]

model.safetensors: downloading bytes:  60% 208M/346M [00:01<00:00, 214MB/s, 18.6MB/s  ]
model.safetensors: reconstructing file:  74% 256M/346M [00:01<00:00, 148MB/s]
model.safetensors: downloading bytes: 100% 231M/231M [00:01<00:00, 125MB/s, 21.2MB/s  ]
model.safetensors: reconstructing file: 100% 346M/346M [00:01<00:00, 187MB/s, 32.7MB/s  ]
Activity model: MCG-NJU/videomae-base-finetuned-kinetics on cuda (16 frames @ 8.0fps, top-5)
[1/40] anomaly/Assault/Assault001_x264.mp4
  39 windows (video 30.00fps)
[2/40] anomaly/Assault/Assault002_x264.mp4
  40 windows (video 30.00fps)
[3/40] anomaly/Assault/Assault003_x264.mp4
  70 windows (video 30.00fps)
[4/40] anomaly/Assault/Assault004_x264.mp4
  51 windows (video 30.00fps)
[5/40] anomaly/Assault/Assault005_x264.mp4
  20 windows (video 30.00fps)
[6/40] anomaly/Fighting/Fighting002_x264.mp4
  42 windows (video 30.00fps)
[7/40

In [ ]:
!python -m src.pipeline.fuse_evidence


Fused 40 videos: 0 detections + 1931 activities + 1931 events -> 1931 records, 40 incidents in 0.6s -> /content/Final/Final/Final/Final/data/evidence/fused
Top video scores:
  anomaly/Assault/Assault004_x264.mp4: 0.9198 {'strength': 0.7995, 'persistence': 1.0, 'concentration': 1.0, 'agreement': 1.0}
  normal/Normal_Videos_050_x264.mp4: 0.9198 {'strength': 0.7996, 'persistence': 1.0, 'concentration': 1.0, 'agreement': 1.0}
  normal/Normal_Videos_129_x264.mp4: 0.9197 {'strength': 0.7992, 'persistence': 1.0, 'concentration': 1.0, 'agreement': 1.0}


In [ ]:
import torch, time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
x = torch.randn(2000, 2000, device=device)

print("Keeping GPU warm... interrupt (stop button) when you're done.")
try:
    while True:
        x = x @ x
        torch.cuda.synchronize()
        time.sleep(5)   # throttle so it's not pointlessly burning compute
except KeyboardInterrupt:
    print("Stopped.")

Keeping GPU warm... interrupt (stop button) when you're done.
Stopped.


In [ ]:
!find /content -maxdepth 6 -type f -path "*src/pipeline/extract_evidence.py" 2>/dev/null

/content/Final/src/pipeline/extract_evidence.py
/content/Final/Final/src/pipeline/extract_evidence.py
/content/Final/Final/Final/src/pipeline/extract_evidence.py


In [ ]:
!pip install -q ultralytics


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 7.7 MB/s eta 0:00:00


In [ ]:
# Cell A — lock into the correct root + verify (no inference, no reinstall)
%cd /content/Final/Final/Final/Final
!pwd && git log --oneline -1 && ls src/pipeline/extract_evidence.py src/pipeline/detect.py src/evidence/fusion.py config/config.example.yaml data/experiments/phase2_subset.json
!echo "ANOMALY=$CCTV_ANOMALY_VIDEOS_DIR" && echo "NORMAL=$CCTV_NORMAL_VIDEOS_DIR"
!ls data/evidence/phase2b/activities.json data/evidence/phase2c_ucf/ucf_events.json && ls data/models/yolo11n.pt 2>&1

/content/Final/Final/Final/Final
/content/Final/Final/Final/Final
87183ec (HEAD -> master, origin/master, origin/HEAD) Phase 2D: evidence fusion layer (alignment, incidents, scores) + tests
config/config.example.yaml	     src/pipeline/detect.py
data/experiments/phase2_subset.json  src/pipeline/extract_evidence.py
src/evidence/fusion.py
ANOMALY=/content/drive/MyDrive/Anomaly-Videos-Part-1/Anomaly-Videos-Part-1
NORMAL=/content/drive/MyDrive/Normal_Videos_for_Event_Recognition/Normal_Videos_for_Event_Recognition
data/evidence/phase2b/activities.json
data/evidence/phase2c_ucf/ucf_events.json
ls: cannot access 'data/models/yolo11n.pt': No such file or directory


In [ ]:
%cd /content/Final/Final/Final/Final
!pwd
!ls -lh yolo11n.pt
!find . -maxdepth 2 -name yolo11n.pt 2>/dev/null

/content/Final/Final/Final/Final
/content/Final/Final/Final/Final
-rw-r--r-- 1 root root 5.4M Sep 14 17:19 yolo11n.pt
./yolo11n.pt


In [ ]:
!mkdir -p data/models
!cp yolo11n.pt data/models/yolo11n.pt
!ls -lh data/models/yolo11n.pt
!python -c "from pathlib import Path;p=Path('data/models/yolo11n.pt');print(p.stat().st_size,'bytes',p.exists())"

-rw-r--r-- 1 root root 5.4M Sep 14 17:20 data/models/yolo11n.pt
5613764 bytes True


In [ ]:
# Cell C — run ONLY Phase 2A for the existing 40-video subset (T4 cuda)
!python -m src.pipeline.extract_evidence --device cuda
# Do NOT add: --skip-detection --limit-videos --max-frames --subset --output-dir
# Do NOT run: inventory / select_subset / extract_activity / extract_ucf_events / training / fuse_evidence

Detector: yolo11n on cuda
[1/40] anomaly/Assault/Assault001_x264.mp4
  83 frames @ 1.0fps (video 30.00fps), 161 detections
[2/40] anomaly/Assault/Assault002_x264.mp4
  85 frames @ 1.0fps (video 30.00fps), 351 detections
[3/40] anomaly/Assault/Assault003_x264.mp4
  149 frames @ 1.0fps (video 30.00fps), 140 detections
[4/40] anomaly/Assault/Assault004_x264.mp4
  108 frames @ 1.0fps (video 30.00fps), 117 detections
[5/40] anomaly/Assault/Assault005_x264.mp4
  41 frames @ 1.0fps (video 30.00fps), 21 detections
[6/40] anomaly/Fighting/Fighting002_x264.mp4
  90 frames @ 1.0fps (video 30.00fps), 9 detections
[7/40] anomaly/Fighting/Fighting003_x264.mp4
  104 frames @ 1.0fps (video 30.00fps), 382 detections
[8/40] anomaly/Fighting/Fighting004_x264.mp4
  560 frames @ 1.0fps (video 30.00fps), 1562 detections
[9/40] anomaly/Fighting/Fighting005_x264.mp4
  60 frames @ 1.0fps (video 30.00fps), 11 detections
[10/40] anomaly/Fighting/Fighting006_x264.mp4
  32 frames @ 1.0fps (video 30.00fps), 113 det

In [ ]:
!python -m src.pipeline.fuse_evidence



Fused 40 videos: 11165 detections + 1931 activities + 1931 events -> 1931 records, 40 incidents in 2.0s -> /content/Final/Final/Final/Final/data/evidence/fused
Top video scores:
  anomaly/Assault/Assault004_x264.mp4: 0.9998 {'strength': 0.9995, 'persistence': 1.0, 'concentration': 1.0, 'agreement': 1.0}
  normal/Normal_Videos_050_x264.mp4: 0.9998 {'strength': 0.9996, 'persistence': 1.0, 'concentration': 1.0, 'agreement': 1.0}
  normal/Normal_Videos_129_x264.mp4: 0.9997 {'strength': 0.9992, 'persistence': 1.0, 'concentration': 1.0, 'agreement': 1.0}


In [ ]:
!mkdir -p /content/drive/MyDrive/Crime_CCTV_Project/Phase2

In [ ]:
!cp -r /content/Final/Final/Final/Final/data/evidence \
      /content/drive/MyDrive/Crime_CCTV_Project/Phase2/

In [ ]:
!mkdir -p /content/drive/MyDrive/Crime_CCTV_Project/Phase2/experiments

!cp /content/Final/Final/Final/Final/data/experiments/phase2_subset.json \
   /content/drive/MyDrive/Crime_CCTV_Project/Phase2/experiments/

In [ ]:
!mkdir -p /content/drive/MyDrive/Crime_CCTV_Project/Phase2/inventory

!cp -r /content/Final/Final/Final/Final/data/inventory/* \
   /content/drive/MyDrive/Crime_CCTV_Project/Phase2/inventory/

In [ ]:
!find /content/drive/MyDrive/Crime_CCTV_Project/Phase2 -type f | sort

/content/drive/MyDrive/Crime_CCTV_Project/Phase2/evidence/fused/evidence.json
/content/drive/MyDrive/Crime_CCTV_Project/Phase2/evidence/fused/incidents.json
/content/drive/MyDrive/Crime_CCTV_Project/Phase2/evidence/fused/manifest.json
/content/drive/MyDrive/Crime_CCTV_Project/Phase2/evidence/fused/video_scores.json
/content/drive/MyDrive/Crime_CCTV_Project/Phase2/evidence/phase2b/activities.json
/content/drive/MyDrive/Crime_CCTV_Project/Phase2/evidence/phase2b/manifest.json
/content/drive/MyDrive/Crime_CCTV_Project/Phase2/evidence/phase2b/videos.json
/content/drive/MyDrive/Crime_CCTV_Project/Phase2/evidence/phase2c_ucf/manifest.json
/content/drive/MyDrive/Crime_CCTV_Project/Phase2/evidence/phase2c_ucf/ucf_events.json
/content/drive/MyDrive/Crime_CCTV_Project/Phase2/evidence/phase2c_ucf/videos.json
/content/drive/MyDrive/Crime_CCTV_Project/Phase2/evidence/phase2/detections.json
/content/drive/MyDrive/Crime_CCTV_Project/Phase2/evidence/phase2/manifest.json
/content/drive/MyDrive/Crime_CC

In [ ]:
import json

path = "data/evidence/fused/evidence.json"

with open(path, "r", encoding="utf-8") as f:
    evidence = json.load(f)

print("Fused records:", len(evidence))
print("First evidence ID:", evidence[0]["evidence_id"])
print("First video:", evidence[0]["video_id"])

Fused records: 1931
First evidence ID: anomaly/Assault/Assault001_x264.mp4:f0
First video: anomaly/Assault/Assault001_x264.mp4


In [ ]:
!git pull

remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 7 (delta 4), reused 7 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (7/7), 5.13 KiB | 2.56 MiB/s, done.
From https://github.com/iesxz-c/Final
   87183ec..d6b5745  master     -> origin/master
Updating 87183ec..d6b5745
Fast-forward
 src/pipeline/retrieve_evidence.py | 206 ++++++++++++++++++++++++++++++++++++++
 tests/test_retrieve.py            | 192 +++++++++++++++++++++++++++++++++++
 2 files changed, 398 insertions(+)
 create mode 100644 src/pipeline/retrieve_evidence.py
 create mode 100644 tests/test_retrieve.py


In [ ]:
!python -m src.pipeline.retrieve_evidence --query "Assault" --limit 10

[anomaly/Assault/Assault004_x264.mp4] anomaly/Assault/Assault004_x264.mp4:f27 57.6-59.6s | surveillance_event | Assault | conf=0.9992 | anomaly/Assault/Assault004_x264.mp4@t=57.6s-59.6s
[anomaly/Assault/Assault004_x264.mp4] anomaly/Assault/Assault004_x264.mp4:f26 55.466-57.466s | surveillance_event | Assault | conf=0.9991 | anomaly/Assault/Assault004_x264.mp4@t=55.466s-57.466s
[anomaly/Assault/Assault004_x264.mp4] anomaly/Assault/Assault004_x264.mp4:f40 85.333-87.333s | surveillance_event | Assault | conf=0.999 | anomaly/Assault/Assault004_x264.mp4@t=85.333s-87.333s
[anomaly/Assault/Assault004_x264.mp4] anomaly/Assault/Assault004_x264.mp4:f28 59.733-61.733s | surveillance_event | Assault | conf=0.9988 | anomaly/Assault/Assault004_x264.mp4@t=59.733s-61.733s
[anomaly/Assault/Assault004_x264.mp4] anomaly/Assault/Assault004_x264.mp4:f21 44.8-46.8s | surveillance_event | Assault | conf=0.9986 | anomaly/Assault/Assault004_x264.mp4@t=44.8s-46.8s
[anomaly/Assault/Assault004_x264.mp4] anomaly/A

In [ ]:
!python -m src.pipeline.retrieve_evidence \
    --query "person" \
    --video-id "anomaly/Assault/Assault001_x264.mp4" \
    --limit 10

[anomaly/Assault/Assault001_x264.mp4] anomaly/Assault/Assault001_x264.mp4:f36 76.8-78.8s | object | person | conf=0.9583 | anomaly/Assault/Assault001_x264.mp4@t=78.0s
[anomaly/Assault/Assault001_x264.mp4] anomaly/Assault/Assault001_x264.mp4:f37 78.934-80.934s | object | person | conf=0.9576 | anomaly/Assault/Assault001_x264.mp4@t=79.0s
[anomaly/Assault/Assault001_x264.mp4] anomaly/Assault/Assault001_x264.mp4:f37 78.934-80.934s | object | person | conf=0.9573 | anomaly/Assault/Assault001_x264.mp4@t=80.0s
[anomaly/Assault/Assault001_x264.mp4] anomaly/Assault/Assault001_x264.mp4:f38 81.067-82.8s | object | person | conf=0.9551 | anomaly/Assault/Assault001_x264.mp4@t=82.0s
[anomaly/Assault/Assault001_x264.mp4] anomaly/Assault/Assault001_x264.mp4:f36 76.8-78.8s | object | person | conf=0.9541 | anomaly/Assault/Assault001_x264.mp4@t=77.0s
[anomaly/Assault/Assault001_x264.mp4] anomaly/Assault/Assault001_x264.mp4:f35 74.667-76.667s | object | person | conf=0.954 | anomaly/Assault/Assault001_x2

In [ ]:
!python -m src.pipeline.retrieve_evidence \
    --query "Fighting" \
    --source surveillance_event \
    --start-time 12.0 \
    --end-time 14.0 \
    --limit 10

[anomaly/Fighting/Fighting004_x264.mp4] anomaly/Fighting/Fighting004_x264.mp4:f5 10.667-12.667s | surveillance_event | Fighting | conf=0.908 | anomaly/Fighting/Fighting004_x264.mp4@t=10.667s-12.667s
[anomaly/Fighting/Fighting004_x264.mp4] anomaly/Fighting/Fighting004_x264.mp4:f6 12.8-14.8s | surveillance_event | Fighting | conf=0.8958 | anomaly/Fighting/Fighting004_x264.mp4@t=12.8s-14.8s
[normal/Normal_Videos_150_x264.mp4] normal/Normal_Videos_150_x264.mp4:f6 12.8-14.8s | surveillance_event | Fighting | conf=0.0091 | normal/Normal_Videos_150_x264.mp4@t=12.8s-14.8s
[anomaly/Assault/Assault003_x264.mp4] anomaly/Assault/Assault003_x264.mp4:f6 12.8-14.8s | surveillance_event | Fighting | conf=0.0039 | anomaly/Assault/Assault003_x264.mp4@t=12.8s-14.8s
4 hit(s) for query 'Fighting' over 1931 records (/content/Final/Final/Final/Final/data/evidence/fused/evidence.json)


In [ ]:
!python -m src.pipeline.retrieve_evidence \
    --query "Assault" \
    --limit 20 \
    --output /content/phase3a_assault_hits.json

[anomaly/Assault/Assault004_x264.mp4] anomaly/Assault/Assault004_x264.mp4:f27 57.6-59.6s | surveillance_event | Assault | conf=0.9992 | anomaly/Assault/Assault004_x264.mp4@t=57.6s-59.6s
[anomaly/Assault/Assault004_x264.mp4] anomaly/Assault/Assault004_x264.mp4:f26 55.466-57.466s | surveillance_event | Assault | conf=0.9991 | anomaly/Assault/Assault004_x264.mp4@t=55.466s-57.466s
[anomaly/Assault/Assault004_x264.mp4] anomaly/Assault/Assault004_x264.mp4:f40 85.333-87.333s | surveillance_event | Assault | conf=0.999 | anomaly/Assault/Assault004_x264.mp4@t=85.333s-87.333s
[anomaly/Assault/Assault004_x264.mp4] anomaly/Assault/Assault004_x264.mp4:f28 59.733-61.733s | surveillance_event | Assault | conf=0.9988 | anomaly/Assault/Assault004_x264.mp4@t=59.733s-61.733s
[anomaly/Assault/Assault004_x264.mp4] anomaly/Assault/Assault004_x264.mp4:f21 44.8-46.8s | surveillance_event | Assault | conf=0.9986 | anomaly/Assault/Assault004_x264.mp4@t=44.8s-46.8s
[anomaly/Assault/Assault004_x264.mp4] anomaly/A

In [ ]:
!python -m src.pipeline.retrieve_evidence --query "person" --video-id "anomaly/Assault/Assault001_x264.mp4" --limit 10

[anomaly/Assault/Assault001_x264.mp4] anomaly/Assault/Assault001_x264.mp4:f36 76.8-78.8s | object | person | conf=0.9583 | anomaly/Assault/Assault001_x264.mp4@t=78.0s
[anomaly/Assault/Assault001_x264.mp4] anomaly/Assault/Assault001_x264.mp4:f37 78.934-80.934s | object | person | conf=0.9576 | anomaly/Assault/Assault001_x264.mp4@t=79.0s
[anomaly/Assault/Assault001_x264.mp4] anomaly/Assault/Assault001_x264.mp4:f37 78.934-80.934s | object | person | conf=0.9573 | anomaly/Assault/Assault001_x264.mp4@t=80.0s
[anomaly/Assault/Assault001_x264.mp4] anomaly/Assault/Assault001_x264.mp4:f38 81.067-82.8s | object | person | conf=0.9551 | anomaly/Assault/Assault001_x264.mp4@t=82.0s
[anomaly/Assault/Assault001_x264.mp4] anomaly/Assault/Assault001_x264.mp4:f36 76.8-78.8s | object | person | conf=0.9541 | anomaly/Assault/Assault001_x264.mp4@t=77.0s
[anomaly/Assault/Assault001_x264.mp4] anomaly/Assault/Assault001_x264.mp4:f35 74.667-76.667s | object | person | conf=0.954 | anomaly/Assault/Assault001_x2

In [ ]:
!python -m src.pipeline.retrieve_evidence \
    --query "Fighting" \
    --source surveillance_event \
    --start-time 12.0 \
    --end-time 14.0 \
    --limit 10

[anomaly/Fighting/Fighting004_x264.mp4] anomaly/Fighting/Fighting004_x264.mp4:f5 10.667-12.667s | surveillance_event | Fighting | conf=0.908 | anomaly/Fighting/Fighting004_x264.mp4@t=10.667s-12.667s
[anomaly/Fighting/Fighting004_x264.mp4] anomaly/Fighting/Fighting004_x264.mp4:f6 12.8-14.8s | surveillance_event | Fighting | conf=0.8958 | anomaly/Fighting/Fighting004_x264.mp4@t=12.8s-14.8s
[normal/Normal_Videos_150_x264.mp4] normal/Normal_Videos_150_x264.mp4:f6 12.8-14.8s | surveillance_event | Fighting | conf=0.0091 | normal/Normal_Videos_150_x264.mp4@t=12.8s-14.8s
[anomaly/Assault/Assault003_x264.mp4] anomaly/Assault/Assault003_x264.mp4:f6 12.8-14.8s | surveillance_event | Fighting | conf=0.0039 | anomaly/Assault/Assault003_x264.mp4@t=12.8s-14.8s
4 hit(s) for query 'Fighting' over 1931 records (/content/Final/Final/Final/Final/data/evidence/fused/evidence.json)
